In [1]:
import torch
import torch.utils.data

In [2]:
import scanpy as sc
import squidpy as sq

from pathlib import Path

from gctsvg import visualization, postprocessing, utils
from gctsvg.preprocessing import st
from gctsvg.model.celltypeembedding import ExpressionDecomposerDriver

/opt/conda/envs/ctSVG/lib/python3.12/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 13.0.0. Please consider upgrading.
  warnings.warn(
/opt/conda/envs/ctSVG/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [3]:
deconv_dir = Path('data_deconvolution/Human Kidney Cancer')

In [4]:
adata_st = sc.read(deconv_dir / 'visium_merge_inter_upload_PD47171.h5ad')

In [5]:
adata_st.obsm['spatial']

array([[7186, 6116],
       [2493, 6999],
       [1419, 7293],
       ...,
       [3794, 6901],
       [3115, 7097],
       [2946, 5622]])

In [6]:
adata_st

AnnData object with n_obs × n_vars = 2048 × 991
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'donor_id', 'region', 'B-cell', 'EC', 'Epi_PT', 'Epi_non-PT', 'Fibro', 'Mast', 'Myeloid', 'NK', 'Plasma', 'RCC', 'T-cell', 'pDC'
    var: 'ENSEMBL', 'feature_types', 'genome', 'mt', 'n_cells_by_counts-6800STDY12499411', 'mean_counts-6800STDY12499411', 'log1p_mean_counts-6800STDY12499411', 'pct_dropout_by_counts-6800STDY12499411', 'total_counts-6800STDY12499411', 'log1p_total_counts-6800STDY12499411', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    uns: 'cell_types', 'hvg', 'log1p', 'overlap_genes', 'sample_colors', 'spatial'
    obsm: 'adj', 'distance_matrix', 'emb_sp', 'feat', 'feat_a', 'graph_neigh', 'label_CSL'

In [7]:
adata_st.obsm['spatial']

array([[7186, 6116],
       [2493, 6999],
       [1419, 7293],
       ...,
       [3794, 6901],
       [3115, 7097],
       [2946, 5622]])

In [8]:
cell_types = adata_st.uns['cell_types']
num_cell_types = len(adata_st.uns['cell_types'])
cell_type_encoding_size = 200

device = "cuda:1"
num_epochs=1200

In [9]:
adata_st[adata_st.obs['Epi_non-PT']>1e-4]

View of AnnData object with n_obs × n_vars = 1181 × 991
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'donor_id', 'region', 'B-cell', 'EC', 'Epi_PT', 'Epi_non-PT', 'Fibro', 'Mast', 'Myeloid', 'NK', 'Plasma', 'RCC', 'T-cell', 'pDC'
    var: 'ENSEMBL', 'feature_types', 'genome', 'mt', 'n_cells_by_counts-6800STDY12499411', 'mean_counts-6800STDY12499411', 'log1p_mean_counts-6800STDY12499411', 'pct_dropout_by_counts-6800STDY12499411', 'total_counts-6800STDY12499411', 'log1p_total_counts-6800STDY12499411', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    uns: 'cell_types', 'hvg', 'log1p', 'overlap_genes', 'sample_colors', 'spatial'
    obsm: 'adj', 'distance_matrix', 'emb_sp', 'feat', 'feat_a', 'graph_neigh', 'la

In [10]:
data = st.create_graph_data(adata_st)
data = data.to(device)
dataset = [data]

In [11]:
data

Data(x=[2048, 991], edge_index=[2, 6144])

In [12]:
driver = ExpressionDecomposerDriver(adata_st.n_vars, num_cell_types, cell_type_encoding_size, device=device)

In [13]:
adata_st.n_obs

2048

In [14]:
losses = driver.train(dataset, batch_size=adata_st.n_obs, num_epochs=num_epochs)

  0%|          | 0/1200 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: split_with_sizes expects split_sizes to sum exactly to 991 (input tensor's size at dimension 1), but got split_sizes=[991, 12]

In [ ]:
len(losses)

In [ ]:
visualization.show_training_curve(range(1,num_epochs + 1), [losses], ["training loss"])

In [ ]:
dataset

In [ ]:
decomposition, reconstruction = driver.get_decomposition(dataset, batch_size = adata_st.n_obs)

In [ ]:
adata_decomposition, adata_reconstruction = postprocessing.build_output_objects(adata_st, decomposition, reconstruction, cell_types)

In [ ]:
adata_st

In [ ]:
adata_decomposition.keys()

In [ ]:
adata_decomposition['B-cell'].X

In [ ]:
adata_st.var

In [ ]:
adata_st.var_names

In [ ]:
adata_reconstruction.raw

In [ ]:
gene = "ACE2"
gene_min, gene_max = utils.get_gene_minmax(adata_st, gene)

In [ ]:
gene_min, gene_max

In [ ]:
sq.pl.spatial_scatter(adata_st, color=gene, vmin=gene_min, vmax=gene_max, title=f"{gene} (original)" )

In [ ]:
sq.pl.spatial_scatter(adata_reconstruction, color=gene, title=f"{gene} (reconstruction)")

In [ ]:
adata_st.obs[cell_types].max()

In [ ]:
visualization.display_cell_deconvolution(adata_st, cell_types)

In [ ]:
for cell_type in cell_types:
    sq.pl.spatial_scatter(adata_decomposition[cell_type], color=gene, title=f"{gene} ({cell_type})")

In [ ]:
adata_st.obsm['spatial']